# Multi-modal Model V2.5 - Fast GRU & Y-Flip Augmentation

이 노트북은 `MultiModal_v2_4`의 속도 이슈를 해결하고, 데이터 증강을 통해 성능을 극대화하는 버전입니다.

## 주요 변경 사항 (V2.5)

1.  **GRU Architecture (Speed)**:
    - LSTM 대신 구조가 단순한 **GRU (Gated Recurrent Unit)**를 사용합니다.
    - 파라미터 수가 적어 학습 속도가 빠르고, 적은 데이터셋에서 과적합 위험이 적습니다.
2.  **Random Y-Flip Augmentation (Robustness)**:
    - 축구장은 Y축(가로 중심선) 기준으로 대칭입니다.
    - 학습 시 **50% 확률로 위아래를 뒤집어(Flip)** 데이터를 2배로 늘리는 효과를 냅니다.
3.  **Visualization Revert (Efficiency)**:
    - v2.4의 속도 저하 원인이었던 Thick Trajectory를 제거하고, 다시 **1px 선 그리기**로 복귀하여 빠른 데이터 로딩 속도를 확보합니다.

## 모델 개요
- **Feature**: v2.2와 동일 (안정적 베이스라인)
- **Model**: CNN + **Bi-GRU** + Fusion
- **Augmentation**: Random Y-Flip (Train Only)

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import GroupKFold
import os
import sys
import random
from tqdm.auto import tqdm
import torch.nn.functional as F

# [변경] V2.5 Preprocessor (v2.2 로직과 동일) 임포트
sys.path.append(os.getcwd())
from src.preprocessing_multimodal import FootballPreprocessorMultimodal

# models 폴더 생성
os.makedirs("models", exist_ok=True)

# 시드 고정
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

SEED = 42
seed_everything(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치 (Device): {DEVICE}")

사용 장치 (Device): cuda


## 1. 데이터 로드 및 전처리 (Preprocessing V2.5)

In [4]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

# Preprocessor V2.5 초기화
preprocessor = FootballPreprocessorMultimodal()
preprocessor.fit(train_df)

input_dim = preprocessor.get_input_dim()
print(f"모델 입력 피처 차원 수: {input_dim}")

print("데이터 변환 중... (Baseline v2.2 Logic)")
train_df_sorted = train_df.sort_values(['game_episode', 'time_seconds'])
train_episodes = preprocessor.transform(train_df_sorted, is_train=True)

groups = []
grouped = train_df_sorted.groupby('game_episode')
for name, group in grouped:
    if len(group) >= 2:
        groups.append(group['game_id'].iloc[0])

print(f"총 에피소드 수: {len(train_episodes)}")

모델 입력 피처 차원 수: 10
데이터 변환 중... (Baseline v2.2 Logic)
총 에피소드 수: 15428


## 2. 이미지 생성 및 증강 (Random Y-Flip)

In [5]:
class MultiModalDataset(Dataset):
    def __init__(self, episodes, img_size=(68, 105), augment=False):
        self.episodes = episodes
        self.H, self.W = img_size
        self.augment = augment # 증강 여부

    def __len__(self):
        return len(self.episodes)

    def _draw_line(self, img, x1, y1, x2, y2, channel, decay_val):
        dist = max(abs(x2 - x1), abs(y2 - y1))
        if dist == 0:
            img[channel, y2, x2] = decay_val
            return
        steps = int(dist * 1.5)
        for i in range(steps + 1):
            t = i / steps
            x = int(x1 + (x2 - x1) * t)
            y = int(y1 + (y2 - y1) * t)
            if 0 <= x < self.W and 0 <= y < self.H:
                img[channel, y, x] = decay_val

    def _generate_image(self, cont_data):
        img = torch.zeros((2, self.H, self.W), dtype=torch.float32)
        seq_len = cont_data.shape[0]
        
        xs = cont_data[:, 0]
        ys = cont_data[:, 1]
        
        x_idxs = (xs * (self.W - 1)).round().astype(int).clip(0, self.W - 1)
        y_idxs = (ys * (self.H - 1)).round().astype(int).clip(0, self.H - 1)
        
        for t in range(seq_len):
            px, py = x_idxs[t], y_idxs[t]
            img[0, py, px] = 1.0
            decay_val = (t + 1) / seq_len
            if t > 0:
                prev_px, prev_py = x_idxs[t-1], y_idxs[t-1]
                self._draw_line(img, prev_px, prev_py, px, py, 1, decay_val)
            else:
                img[1, py, px] = decay_val
        return img

    def __getitem__(self, idx):
        data = self.episodes[idx]
        
        cont = data['cont'].copy() # 원본 보존을 위해 copy
        target = data['target'].copy()
        
        # [수정] V8 Vector Features
        # Cols: 0:start_x, 1:start_y, 2:end_x_prev, 3:end_y_prev, 4:dx_prev, 5:dy_prev
        
        if self.augment and random.random() < 0.5:
            # Y-Flip
            cont[:, 1] = 1.0 - cont[:, 1] # start_y
            cont[:, 3] = 1.0 - cont[:, 3] # end_y_prev
            cont[:, 5] = -cont[:, 5]      # dy_prev (Vector Flip)
            target[1] = 1.0 - target[1]   # Target Flip
        
        img = self._generate_image(cont)
        cont_tensor = torch.tensor(cont, dtype=torch.float32)
        cat_tensor = torch.tensor(data['cat'], dtype=torch.long)
        target_tensor = torch.tensor(target, dtype=torch.float32)
        
        return img, cont_tensor, cat_tensor, target_tensor

def multimodal_collate_fn(batch):
    imgs, conts, cats, targets = zip(*batch)
    imgs_batched = torch.stack(imgs, dim=0) 
    lengths = torch.tensor([len(c) for c in conts], dtype=torch.long)
    conts_padded = pad_sequence(conts, batch_first=True)
    cats_padded = pad_sequence(cats, batch_first=True)
    targets = torch.stack(targets, dim=0)
    return imgs_batched, conts_padded, cats_padded, lengths, targets

## 3. 모델 아키텍처 (GRU 적용)

In [6]:
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        x_out = self.conv(x_cat)
        return self.sigmoid(x_out)

class ImprovedCNN(nn.Module):
    def __init__(self):
        super(ImprovedCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(2, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),
        )
        self.sa = SpatialAttention()
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, 128)
        
    def forward(self, x):
        x = self.features(x)
        sa_map = self.sa(x)
        x = x * sa_map
        x = self.pool(x).flatten(1)
        x = F.relu(self.fc(x))
        return x

class LSTMAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(LSTMAttention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)
    def forward(self, rnn_output):
        attn_weights = torch.softmax(self.attention(rnn_output), dim=1)
        return torch.sum(attn_weights * rnn_output, dim=1)

class MultiModalNetV2(nn.Module):
    def __init__(self, input_dim_cont, num_types, num_results, gru_hidden=128):
        super(MultiModalNetV2, self).__init__()
        self.cnn = ImprovedCNN()
        self.type_emb = nn.Embedding(num_types, 8)
        self.result_emb = nn.Embedding(num_results, 8)
        
        total_input_dim = input_dim_cont + 8 + 8
        
        # [변경] LSTM -> GRU
        self.gru = nn.GRU(
            input_size=total_input_dim,
            hidden_size=gru_hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.1
        )
        self.gru_attn = LSTMAttention(gru_hidden * 2)
        self.gru_fc = nn.Linear(gru_hidden * 2, 128)
        
        self.fusion_fc = nn.Sequential(
            nn.BatchNorm1d(256), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, img, cont, cat, lengths):
        img_feat = self.cnn(img)
        emb_type = self.type_emb(cat[:, :, 0])
        emb_result = self.result_emb(cat[:, :, 1])
        x_seq = torch.cat([cont, emb_type, emb_result], dim=2)
        
        packed = pack_padded_sequence(x_seq, lengths.cpu(), batch_first=True, enforce_sorted=False)
        # GRU returns: output, h_n
        packed_out, _ = self.gru(packed)
        gru_out, _ = pad_packed_sequence(packed_out, batch_first=True)
        
        gru_ctx = self.gru_attn(gru_out)
        seq_feat = F.relu(self.gru_fc(gru_ctx))
        concat_feat = torch.cat([img_feat, seq_feat], dim=1)
        return self.fusion_fc(concat_feat)

class EuclideanLoss(nn.Module):
    def __init__(self):
        super(EuclideanLoss, self).__init__()
    def forward(self, pred, target):
        pred_real = pred * torch.tensor([105.0, 68.0], device=pred.device)
        target_real = target * torch.tensor([105.0, 68.0], device=target.device)
        return torch.mean(torch.sqrt(torch.sum((pred_real - target_real)**2, dim=1) + 1e-6))

## 4. 학습

In [7]:
def train_multimodal(train_episodes, groups, n_splits=5, epochs=30, batch_size=64, lr=0.001):
    gkf = GroupKFold(n_splits=n_splits)
    
    input_dim_cont = preprocessor.get_input_dim()
    num_types, num_results = preprocessor.get_num_classes()
    
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(gkf.split(train_episodes, groups=groups)):
        print(f"\n=== Fold {fold+1}/{n_splits} ===")
        
        train_sub = [train_episodes[i] for i in train_idx]
        val_sub = [train_episodes[i] for i in val_idx]
        
        # [핵심] Train Dataset에만 Augmentation 적용
        train_dataset = MultiModalDataset(train_sub, augment=True)
        val_dataset = MultiModalDataset(val_sub, augment=False)
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=multimodal_collate_fn)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=multimodal_collate_fn)
        
        model = MultiModalNetV2(input_dim_cont, num_types, num_results).to(DEVICE)
        criterion = EuclideanLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr) 
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
        
        best_dist = float('inf')
        best_state = None
        
        for epoch in range(epochs):
            model.train()
            train_loss = 0
            for imgs, cont, cat, lengths, target in train_loader:
                imgs, cont, cat, lengths, target = imgs.to(DEVICE), cont.to(DEVICE), cat.to(DEVICE), lengths.to(DEVICE), target.to(DEVICE)
                optimizer.zero_grad()
                pred = model(imgs, cont, cat, lengths)
                loss = criterion(pred, target)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            
            train_loss /= len(train_loader)
            
            model.eval()
            val_dists = []
            with torch.no_grad():
                for imgs, cont, cat, lengths, target in val_loader:
                    imgs, cont, cat, lengths, target = imgs.to(DEVICE), cont.to(DEVICE), cat.to(DEVICE), lengths.to(DEVICE), target.to(DEVICE)
                    pred = model(imgs, cont, cat, lengths)
                    pred_real = pred.cpu().numpy() * np.array([105.0, 68.0])
                    target_real = target.cpu().numpy() * np.array([105.0, 68.0])
                    val_dists.extend(np.sqrt(np.sum((pred_real - target_real)**2, axis=1)))
            
            mean_dist = np.mean(val_dists)
            scheduler.step(mean_dist)
            print(f"Epoch {epoch+1}: Train Loss {train_loss:.4f}, Val Dist {mean_dist:.4f}")
            
            if mean_dist < best_dist:
                best_dist = mean_dist
                best_state = model.state_dict()
                torch.save(best_state, f"models/multimodal_v8_fold_{fold+1}.pth")
                print(f"  -> Saved Best Model (Dist: {best_dist:.4f})")
        
        print(f"Fold {fold+1} Finished. Best Valid Dist: {best_dist:.4f}")
        fold_scores.append(best_dist)
        
        torch.save(best_state, f"models/multimodal_v8_fold_{fold+1}.pth")
        
    print(f"Average Score: {np.mean(fold_scores):.4f}")

In [8]:
train_multimodal(train_episodes, groups, n_splits=5, epochs=30, batch_size=64, lr=0.001)


=== Fold 1/5 ===
Epoch 1: Train Loss 29.1495, Val Dist 21.6663
Epoch 2: Train Loss 20.2750, Val Dist 20.4546
Epoch 3: Train Loss 19.0221, Val Dist 18.7432
Epoch 4: Train Loss 18.1352, Val Dist 17.8391
Epoch 5: Train Loss 17.7494, Val Dist 17.5760
Epoch 6: Train Loss 17.2015, Val Dist 16.3061
Epoch 7: Train Loss 16.7436, Val Dist 17.0841
Epoch 8: Train Loss 16.6468, Val Dist 15.8945
Epoch 9: Train Loss 16.3154, Val Dist 15.9188
Epoch 10: Train Loss 16.1882, Val Dist 15.6510
Epoch 11: Train Loss 15.9729, Val Dist 15.1736
Epoch 12: Train Loss 15.7373, Val Dist 16.1421
Epoch 13: Train Loss 15.8088, Val Dist 15.8223
Epoch 14: Train Loss 15.4664, Val Dist 15.1802
Epoch 15: Train Loss 15.5019, Val Dist 14.6715
Epoch 16: Train Loss 15.4099, Val Dist 14.8225
Epoch 17: Train Loss 15.0608, Val Dist 15.5357
Epoch 18: Train Loss 15.0175, Val Dist 14.6935
Epoch 19: Train Loss 14.9827, Val Dist 15.2841
Epoch 20: Train Loss 14.6080, Val Dist 14.5987
Epoch 21: Train Loss 14.4156, Val Dist 14.4160
Epoc

## 5. 추론 (Inference)

In [10]:
class TestMultiModalDataset(Dataset):
    def __init__(self, submission_df, preprocessor, img_size=(68, 105)):
        self.submission_df = submission_df
        self.preprocessor = preprocessor
        self.H, self.W = img_size

    def __len__(self):
        return len(self.submission_df)

    def _draw_line(self, img, x1, y1, x2, y2, channel, decay_val):
        dist = max(abs(x2 - x1), abs(y2 - y1))
        if dist == 0:
            img[channel, y2, x2] = decay_val
            return
        steps = int(dist * 1.5)
        for i in range(steps + 1):
            t = i / steps
            x = int(x1 + (x2 - x1) * t)
            y = int(y1 + (y2 - y1) * t)
            if 0 <= x < self.W and 0 <= y < self.H:
                img[channel, y, x] = decay_val

    def _generate_image(self, cont_data):
        img = torch.zeros((2, self.H, self.W), dtype=torch.float32)
        if len(cont_data) == 0: return img
        seq_len = cont_data.shape[0]
        xs = cont_data[:, 0]
        ys = cont_data[:, 1]
        x_idxs = (xs * (self.W - 1)).round().astype(int).clip(0, self.W - 1)
        y_idxs = (ys * (self.H - 1)).round().astype(int).clip(0, self.H - 1)
        for t in range(seq_len):
            px, py = x_idxs[t], y_idxs[t]
            img[0, py, px] = 1.0
            decay_val = (t + 1) / seq_len
            if t > 0:
                prev_px, prev_py = x_idxs[t-1], y_idxs[t-1]
                self._draw_line(img, prev_px, prev_py, px, py, 1, decay_val)
            else:
                img[1, py, px] = decay_val
        return img

    def __getitem__(self, idx):
        row = self.submission_df.iloc[idx]
        raw_path = row['path']
        if raw_path.startswith("./"):
            path = raw_path[2:]
        else:
            path = raw_path
            
        df = pd.read_csv(path)
        episodes = self.preprocessor.transform(df, is_train=False)
        
        if len(episodes) == 0:
            input_dim = self.preprocessor.get_input_dim()
            cont = torch.zeros((1, input_dim), dtype=torch.float32)
            cat = torch.zeros((1, 2), dtype=torch.long)
            img = torch.zeros((2, self.H, self.W), dtype=torch.float32)
        else:
            data = episodes[0]
            cont_np = data['cont']
            img = self._generate_image(cont_np)
            cont = torch.tensor(cont_np, dtype=torch.float32)
            cat = torch.tensor(data['cat'], dtype=torch.long)
        
        return img, cont, cat

def test_mm_collate_fn(batch):
    imgs, conts, cats = zip(*batch)
    imgs_batched = torch.stack(imgs, dim=0)
    lengths = torch.tensor([len(c) for c in conts], dtype=torch.long)
    conts_padded = pad_sequence(conts, batch_first=True)
    cats_padded = pad_sequence(cats, batch_first=True)
    return imgs_batched, conts_padded, cats_padded, lengths

submission = pd.read_csv("sample_submission.csv")
test_meta = pd.read_csv("test.csv")
submission = submission.merge(test_meta, on="game_episode", how="left")

test_dataset = TestMultiModalDataset(submission, preprocessor)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, collate_fn=test_mm_collate_fn)

print("Inference started...")

input_dim_cont = preprocessor.get_input_dim()
num_types, num_results = preprocessor.get_num_classes()

trained_models = []
for i in range(5):
    model = MultiModalNetV2(input_dim_cont, num_types, num_results).to(DEVICE)
    # v2.5 models load
    model.load_state_dict(torch.load(f"models/multimodal_v8_fold_{i+1}.pth"))
    model.eval()
    trained_models.append(model)

preds_x = []
preds_y = []

with torch.no_grad():
    for imgs, cont, cat, lengths in tqdm(test_loader):
        imgs, cont, cat, lengths = imgs.to(DEVICE), cont.to(DEVICE), cat.to(DEVICE), lengths.to(DEVICE)
        batch_preds = np.zeros((cont.size(0), 2))
        for model in trained_models:
            pred = model(imgs, cont, cat, lengths)
            batch_preds += pred.cpu().numpy()
        batch_preds /= 5.0
        batch_preds[:, 0] *= 105.0
        batch_preds[:, 1] *= 68.0
        preds_x.extend(batch_preds[:, 0])
        preds_y.extend(batch_preds[:, 1])

submission['end_x'] = preds_x
submission['end_y'] = preds_y
submission[['game_episode', 'end_x', 'end_y']].to_csv("submission_multimodal.csv", index=False)
print("Saved submission_multimodal.csv")

Inference started...


C:\Users\semic\AppData\Local\Temp\ipykernel_64844\136662553.py:91: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"models/multimodal_v8_fold

  0%|          | 0/19 [00:00<?, ?it/s]

Saved submission_multimodal.csv


In [11]:
result = pd.read_csv("submission_multimodal.csv")
result.head()

,game_episode,end_x,end_y
0,153363_1,63.607921,11.172691
1,153363_2,24.612310,56.282756
2,153363_6,31.860115,64.409105
3,153363_7,52.938785,5.637901
4,153363_8,82.286735,7.873227
